# 3. RAG vectoriel et Graph RAG

Sélection des 5 articles par chaque mode, sur deux questions du test à l'aveugle. Les candidats MultiQuery et HyDE sont relus dans le cache (aucune requête au LLM) ; le reranker `bge-reranker-v2-m3` tourne en local.

_Notebook rejoué avec le code du projet (`graphrag/`, et `app/` du projet vectoriel). Ses sorties sont réelles : il a été exécuté le 10/11/2026 à 03:04 par `scripts/build_notebooks.py`. Aucun appel au LLM n'est fait ici ; Neo4j doit être démarré._

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# candidats de recherche déjà calculés (MultiQuery, HyDE) : aucune requête au LLM
os.environ.setdefault("SEED_CACHE_PATH", str(ROOT / "data" / "cache" / "user_test_candidates.json"))

import pandas as pd
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)

from graphrag import config
config.use_vector_rag()
DATA = config.VECTOR_RAG_DIR / "data" / "eurlex"
RESULTS = ROOT / "results"
print("Projet :", ROOT)
print("Données du projet vectoriel :", DATA)

Projet : C:\Users\hazem\5eme\GraphRAG-juridique
Données du projet vectoriel : C:\Users\hazem\5eme\Advenced-Rag-Agents-main\Advenced-Rag-Agents-main\data\eurlex


In [2]:
from app import config as rag_config
from app.rag.prompts import ANSWER_TEMPLATE

print("Embeddings :", rag_config.EMBEDDING_MODEL)
print("Reranker   :", rag_config.LOCAL_RERANK_MODEL)
print("LLM        :", rag_config.LLM_MODEL, "| modèle rapide :", rag_config.FAST_LLM_MODEL)
print("Articles dans le contexte :", rag_config.TOP_K, "| candidats :", rag_config.CANDIDATES_K)

Embeddings : intfloat/multilingual-e5-base
Reranker   : BAAI/bge-reranker-v2-m3
LLM        : openai/gpt-oss-120b | modèle rapide : openai/gpt-oss-20b
Articles dans le contexte : 5 | candidats : 20


## Sélection des articles, mode par mode

`vector` : cosinus seul. `advanced` (baseline) : MultiQuery + HyDE + BM25, fusion RRF, reranker. `graph_linked` : baseline + graphe, 2 places réservées aux renvois inter-actes de l'article classé 1er.

In [3]:
from graphrag.evaluate import retrieve_fn_for

def selection(question, modes=("vector", "advanced", "graph_linked")):
    lignes = []
    for mode in modes:
        for rang, c in enumerate(retrieve_fn_for(mode)(question, 5), start=1):
            m = c.metadata
            lignes.append({"mode": mode, "rang": rang, "article": f"{m['doc_label']}, art. {m['article_num']}",
                           "titre": m.get("article_title"), "ajoute_par_le_graphe": m.get("graph_added", False),
                           "chemin": (m.get("via") or [""])[0]})
    return pd.DataFrame(lignes)

selection("Quel acte du corpus modifie le règlement 2019/1020 ?")

,mode,rang,article,titre,ajoute_par_le_graphe,chemin
0,vector,1,"Règlement (UE) 2019/1020, art. 38",Modification de la directive 2004/42/CE,False,
1,vector,2,"Règlement (UE) 2019/1020, art. 39",Modification du règlement (CE) no 765/2008,False,
2,vector,3,"Règlement (UE) 2019/818, art. 61",Modifications du règlement (UE) 2019/816,False,
3,vector,4,"Règlement (UE) 2019/818, art. 59",Modifications du règlement (UE) 2018/1726,False,
4,vector,5,"Règlement (UE) 2019/1020, art. 34",Système d'information et de communication,False,
5,advanced,1,"Règlement (UE) 2019/1020, art. 39",Modification du règlement (CE) no 765/2008,False,
6,advanced,2,"Règlement (UE) 2019/1020, art. 38",Modification de la directive 2004/42/CE,False,
7,advanced,3,"Règlement (UE) 2019/1020, art. 34",Système d'information et de communication,False,
8,advanced,4,"Règlement (UE) 2019/1020, art. 43",Comité,False,
9,advanced,5,"Règlement (UE) 2019/1700, art. 20",Modification du règlement (CE) n° 808/2004,False,


In [4]:
selection("Peut-on montrer à un mineur de la publicité fondée sur le profilage ?")

,mode,rang,article,titre,ajoute_par_le_graphe,chemin
0,vector,1,"Règlement (UE) 2022/2065 (DSA), art. 28",Protection des mineurs en ligne,False,
1,vector,2,"Règlement (UE) 2024/900 (règlement publicité politique), art. 18",Exigences spécifiques liées aux techniques de ciblage et aux techniquesde diffusion d'annonces publicitaires dans le contexte de la publicité à caractère politique en ligne,False,
2,vector,3,"Règlement (UE) 2022/2065 (DSA), art. 26",Publicité sur les plateformes en ligne,False,
3,vector,4,"Règlement (UE) 2024/1358, art. 14",Dispositions particulières concernant les mineurs,False,
4,vector,5,"Règlement (UE) 2024/1358, art. 42",Droit à l’information,False,
5,advanced,1,"Règlement (UE) 2022/2065 (DSA), art. 28",Protection des mineurs en ligne,False,
6,advanced,2,"Règlement (UE) 2022/2065 (DSA), art. 26",Publicité sur les plateformes en ligne,False,
7,advanced,3,"Règlement (UE) 2024/900 (règlement publicité politique), art. 18",Exigences spécifiques liées aux techniques de ciblage et aux techniquesde diffusion d'annonces publicitaires dans le contexte de la publicité à caractère politique en ligne,False,
8,advanced,4,"Règlement (UE) 2018/1725 (EUDPR), art. 77","Décision individuelle automatisée, y compris le profilage",False,
9,advanced,5,"Règlement (UE) 2016/679 (RGPD), art. 22","Décision individuelle automatisée, y compris le profilage",False,


## Contexte envoyé au LLM

Le meilleur chunk de chaque article ne contient pas toujours le paragraphe qui répond. Tous les modes envoient donc l'article complet (6 000 caractères au plus) ; la sélection ne change pas.

In [5]:
from graphrag import retriever as graph

question = "Peut-on montrer à un mineur de la publicité fondée sur le profilage ?"
chunks = graph.graph_retrieve(question, 5, mode="graph_linked")
complets = graph.full_article_context(chunks)
pd.DataFrame({"article": [c.metadata["article_id"] for c in chunks],
              "caracteres_meilleur_chunk": [len(c.page_content) for c in chunks],
              "caracteres_article_complet": [len(c.page_content) for c in complets]})

,article,caracteres_meilleur_chunk,caracteres_article_complet
0,32022R2065_art_28,1121,1121
1,32022R2065_art_26,1219,2254
2,32024R0900_art_18,1218,3331
3,32016R0679_art_4,1628,6328
4,32018R1725_art_77,1193,1193


In [6]:
print(complets[0].page_content[:900])

Règlement (UE) 2022/2065 (DSA) — Article 28 — Protection des mineurs en ligne
1. Les fournisseurs de plateformes en ligne accessibles aux mineurs mettent en place des mesures appropriées et proportionnées pour garantir un niveau élevé de protection de la vie privée, de sûreté et de sécurité des mineurs sur leur service. 2. Les fournisseurs de plateformes en ligne ne présentent pas sur leur interface de publicité qui repose sur du profilage, tel qu’il est défini à l’article 4, point 4), du règlement (UE) 2016/679 en utilisant des données à caractère personnel concernant le destinataire du service dès lors qu’ils ont connaissance avec une certitude raisonnable que le destinataire du service est un mineur. 3. Le respect des obligations énoncées dans le présent article n’impose pas aux fournisseurs de plateformes en ligne de traiter des données à caractère personnel supplémentaires afin de d


## Le prompt partagé par tous les modes

In [7]:
print(ANSWER_TEMPLATE)

Tu es un assistant juridique spécialisé dans le {domain}.
Tu réponds à partir des extraits d'articles officiels (EUR-Lex) fournis ci-dessous, et d'eux seuls.

Règles :
1. Appuie chaque affirmation sur un extrait en citant son numéro entre crochets, ex. [1], et nomme l'acte et l'article (ex. « article 17 du RGPD »).
2. Reprends fidèlement les conditions, exceptions, délais, seuils et montants tels qu'ils figurent dans le texte.
3. Si les extraits ne permettent pas de répondre (ou seulement en partie), dis-le explicitement : « Les articles fournis ne permettent pas de répondre à cette question. » N'invente jamais un numéro d'article, un délai, un montant ou une obligation.
4. Réponds uniquement à partir des extraits : tes connaissances propres ne sont pas une source. Si un titre, une date ou un contenu n'y figure pas, dis-le, ou cite seulement le numéro de l'acte tel qu'il apparaît (ex. « directive (UE) 2020/1828 »), sans y ajouter d'intitulé, d'objet ou de surnom.
5. Les guillemets « » 

## Réponses générées (enregistrées)

Produites par `scripts/answer_user_questions.py` avec `gpt-oss-120b`, même chaîne que l'interface. Relues ensuite à la main contre le texte des articles.

In [8]:
answers = json.loads((RESULTS / "answers_utilisateur_20261110_015420.json").read_text(encoding="utf-8"))
c2 = next(r for r in answers["rows"] if r["id"] == "C2")
print("Question :", c2["question"], "\nRéponse attendue :", c2["reference"])
for mode in ("advanced", "graph_linked"):
    print(f"\n--- {mode} (articles : {c2[mode]['selected']})\n{c2[mode]['answer']}")

Question : Quel acte du corpus modifie le règlement 2019/1020 ? 
Réponse attendue : Le règlement (UE) 2024/2847.

--- advanced (articles : ['32019R1020_art_39', '32019R1020_art_38', '32019R1020_art_34', '32019R1020_art_43', '32019R1700_art_20'])
Les articles fournis ne permettent pas de répondre à cette question. Aucun extrait du corpus ne indique qu’un autre acte modifie le règlement (UE) 2019/1020.

--- graph_linked (articles : ['32024R2847_art_66', '32019R1020_art_39', '32019R1020_art_38', '32019R1020_art_40', '32019R1020_art_25'])
Le **Règlement (UE) 2024/2847** (Cyber Resilience Act) modifie le **Règlement (UE) 2019/1020** ; la modification est prévue à l’article 66 du règlement 2024/2847, qui ajoute le point « 72. Règlement (UE) 2024/2847 du Parlement européen et du Conseil » à l’annexe I du règlement (UE) 2019/1020 [1] (Règlement (UE) 2024/2847, article 66).
